goal: run susie with ld calcualted from cojo

Calculate ld for chr 16 using cojo 

-- bfile ../../ALL_chr/plink_b_files/chr16_eur_filtered
--cojo-file ../../reformated_data_for_cojo/cojo_extracted_file.csv
-out ../../cojo/chr16/chr16


icog-bioai@dl:/mnt/hdd_2/rediet/hypothesis-generation-demo/data/susie/gcta/gcta-1.94.3-linux-kernel-3-x86_64$ gcta64 --bfile ../../ALL_chr/plink_b_files/chr16_eur_filtered --chr 16  --cojo-file ../../reformated_data_for_cojo/cojo_extracted_file.csv  --cojo-slct --out ../../cojo/chr16/chr16
*******************************************************************
* Genome-wide Complex Trait Analysis (GCTA)
* version v1.94.1 Linux
* Built at Dec 16 2024 16:35:13, by GCC 8.3
* (C) 2010-present, Yang Lab, Westlake University
* Please report bugs to Jian Yang <jian.yang@westlake.edu.cn>
*******************************************************************
Analysis started at 05:58:54 PST on Mon Feb 24 2025.
Hostname: dl

Accepted options:
--bfile ../../ALL_chr/plink_b_files/chr16_eur_filtered
--chr 16
--cojo-file ../../reformated_data_for_cojo/cojo_extracted_file.csv
--cojo-slct
--out ../../cojo/chr16/chr16


Reading PLINK FAM file from [../../ALL_chr/plink_b_files/chr16_eur_filtered.fam].
2504 individuals to be included from [../../ALL_chr/plink_b_files/chr16_eur_filtered.fam].
Reading PLINK BIM file from [../../ALL_chr/plink_b_files/chr16_eur_filtered.bim].
2415 SNPs to be included from [../../ALL_chr/plink_b_files/chr16_eur_filtered.bim].
2415 SNPs on chromosome 16 are included in the analysis.
Reading PLINK BED file from [../../ALL_chr/plink_b_files/chr16_eur_filtered.bed] in SNP-major format ...
Genotype data for 2504 individuals and 2415 SNPs to be included from [../../ALL_chr/plink_b_files/chr16_eur_filtered.bed].

Reading GWAS summary-level statistics from [../../reformated_data_for_cojo/cojo_extracted_file.csv] ...
GWAS summary statistics of 49937 SNPs read from [../../reformated_data_for_cojo/cojo_extracted_file.csv].
Phenotypic variance estimated from summary statistics of all 49937 SNPs: 22.4081 (variance of logit for case-control studies).
Matching the GWAS meta-analysis results to the genotype data ...
Calculating allele frequencies ...
557 SNP(s) have large difference of allele frequency between the GWAS summary data and the reference sample. These SNPs have been saved in [../../cojo/chr16/chr16.freq.badsnps].
1858 SNPs are matched to the genotype data.
Calculating the variance of SNP genotypes ...

Performing stepwise model selection on 1858 SNPs to select association signals ... (p cutoff = 5e-08; collinearity cutoff = 0.9)
(Assuming complete linkage equilibrium between SNPs which are more than 10Mb away from each other)
5 associated SNPs have been selected.
10 associated SNPs have been selected.
15 associated SNPs have been selected.
20 associated SNPs have been selected.
25 associated SNPs have been selected.
30 associated SNPs have been selected.
Finally, 30 associated SNPs are selected.
Performing joint analysis on all the 30 selected signals ...
Saving the 30 independent signals to [../../cojo/chr16/chr16.jma.cojo] ...
Saving the LD structure of 30 independent signals to [../../cojo/chr16/chr16.ldr.cojo] ...
Saving the conditional analysis results of 1822 remaining SNPs to [../../cojo/chr16/chr16.cma.cojo] ...
(1 SNPs eliminated by backward selection and 5 SNPs filtered by collinearity test are not included in the output)

Analysis finished at 05:58:56 PST on Mon Feb 24 2025
Overall computational time: 1.34 sec.

In [22]:
import pandas as pd
chr16_ldr_cojo=pd.read_csv("../data/susie/cojo/chr16/chr16.ldr.cojo", sep="\t", header=None)
chr16_ldr_cojo=chr16_ldr_cojo.values
chr16_ldr_cojo=chr16_ldr_cojo[:, :-1]
chr16_ldr_cojo.shape

(30, 30)

In [14]:

from rpy2.robjects.packages import importr
import rpy2.robjects as ro
import rpy2.robjects.numpy2ri as numpy2ri
import rpy2.robjects.pandas2ri as pandas2ri
import matplotlib.pyplot as plt
numpy2ri.activate()
pandas2ri.activate()

In [16]:
susieR = importr('susieR')

In [19]:
test_susie_with_cojo_ld_snps= pd.read_csv("../data/susie/cojo/chr16/chr16.jma.cojo", sep="\t")

In [36]:
ro.r('set.seed(123)')
fit = susieR.susie_rss(
    bhat = test_susie_with_cojo_ld_snps["b"].values.reshape(len(chr16_ldr_cojo ), 1),
    shat = test_susie_with_cojo_ld_snps["se"].values.reshape(len(chr16_ldr_cojo ), 1),
    R = chr16_ldr_cojo,
    L = 10,
    n=359983
    
)

In [37]:
credible_sets = susieR.susie_get_cs(fit, coverage=0.95, min_abs_corr=0.5, Xcorr=chr16_ldr_cojo)
print(credible_sets)

$cs
$cs$L1
[1] 19

$cs$L2
[1] 14

$cs$L3
[1] 11

$cs$L4
[1] 13

$cs$L5
[1] 12

$cs$L6
[1] 3

$cs$L7
[1] 25

$cs$L8
[1] 8

$cs$L9
[1] 22

$cs$L10
[1] 21


$purity
    min.abs.corr mean.abs.corr median.abs.corr
L1             1             1               1
L2             1             1               1
L3             1             1               1
L4             1             1               1
L5             1             1               1
L6             1             1               1
L7             1             1               1
L8             1             1               1
L9             1             1               1
L10            1             1               1

$cs_index
 [1]  1  2  3  4  5  6  7  8  9 10

$coverage
 [1] 1.0000000 1.0000000 1.0000000 1.0000000 1.0000000 0.9999999 0.9998370
 [8] 1.0000000 1.0000000 1.0000000

$requested_coverage
[1] 0.95




In [38]:
n_cs = len(credible_sets)
n_cs

5

In [39]:
import numpy as np
cs_list = credible_sets.rx2('cs')
cs_index = [idx for cs in cs_list if cs is not None for idx in cs]
cs_index = np.array(cs_index, dtype=int)
credible_snps = test_susie_with_cojo_ld_snps.iloc[cs_index, :]
print(credible_snps)


    Chr               SNP        bp refA      freq         b        se  \
19   16   16:53830491:T:C  53830491    C  0.432609  0.313572  0.011237   
14   16   16:53814470:C:T  53814470    T  0.129047 -0.118643  0.016640   
11   16   16:29994922:C:T  29994922    T  0.484633  0.126127  0.011189   
13   16  16:53807005:AT:A  53807005    A  0.482868  0.299494  0.011184   
12   16   16:31075175:G:A  31075175    A  0.373145 -0.124927  0.011521   
3    16    16:4932470:G:A   4932470    A  0.437222 -0.068619  0.011281   
25   16   16:70514828:A:C  70514828    C  0.452027  0.079024  0.011236   
8    16   16:24811740:T:C  24811740    C  0.266977 -0.081349  0.012596   
22   16  16:53864764:CG:C  53864764    C  0.322651 -0.081399  0.012050   
21   16  16:53839355:GT:G  53839355    G  0.405865  0.122079  0.012224   

                p       n  freq_geno        bJ     bJ_se             pJ  \
19  2.348400e-171  360693   0.291334 -0.329204  0.024414   1.934000e-41   
14   1.003460e-12  359974   0.10024

calculate ld using plink

In [40]:
!plink \
  --bfile "../data/susie/plink_binary/chr16_eur_filtered" \
  --keep-allele-order \
  --r square \
  --extract ../data/susie/cojo/chr16/chr16.jma.cojo\
  --out ../data/susie/ALL_chr/ld/IBSS_test_sig_locus_mt

!plink \
  --bfile "../data/susie/plink_binary/chr16_eur_filtered" \
  --keep-allele-order \
  --r2 square \
  --extract ../data/susie/cojo/chr16/chr16.jma.cojo \
  --out ../data/susie/ALL_chr/ld/IBSS_test_sig_locus_mt_r2

PLINK v1.90b7.2 64-bit (11 Dec 2023)           www.cog-genomics.org/plink/1.9/
(C) 2005-2023 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to ../data/susie/ALL_chr/ld/IBSS_test_sig_locus_mt.log.
Options in effect:
  --bfile ../data/susie/plink_binary/chr16_eur_filtered
  --extract ../data/susie/cojo/chr16/chr16.jma.cojo
  --keep-allele-order
  --out ../data/susie/ALL_chr/ld/IBSS_test_sig_locus_mt
  --r square

257418 MB RAM detected; reserving 128709 MB for main workspace.
2415 variants loaded from .bim file.
503 people (0 males, 0 females, 503 ambiguous) loaded from .fam.
Ambiguous sex IDs written to
../data/susie/ALL_chr/ld/IBSS_test_sig_locus_mt.nosex .
--extract: 30 variants remaining.
Using up to 27 threads (change this with --threads).
Before main variant filters, 503 founders and 0 nonfounders present.
Calculating allele frequencies... 1011121314151617181920212223242526272829303132333435363738394041424344454647484950515253545556575859606162636465666768

In [41]:
ld_r = pd.read_csv("../data/susie/ALL_chr/ld/IBSS_test_sig_locus_mt.ld", sep="\t", header=None)
R_df = ld_r.values
ld2 = pd.read_csv("../data/susie/ALL_chr/ld/IBSS_test_sig_locus_mt_r2.ld",sep="\t",header=None)
R_df2 = ld2.values
R_df.shape

(30, 30)